In [ ]:
from matplotlib_inline.backend_inline import set_matplotlib_formats
from matplotlib.pyplot import cm
from os import listdir as ls
from IPython.display import display, Markdown
import pycountry_convert as pc
import yaml as yml
import matplotlib.pyplot as plt

from emu_renewal.constants import OUTPUTS_PATH, DATA_PATH, MOB_COLOURS
from emu_renewal.priors import get_standard_priors
from emu_renewal.outputs import get_param_vals_by_analysis, get_prop_improve
from emu_renewal.plotting import ANALYSIS_NAMES, plot_kde_comparison, get_param_medians, AN_ABBREVS, plot_world_country_outline, plot_prop_improve
from emu_renewal.utils import get_countries_by_continent

set_matplotlib_formats("svg")

In [ ]:
job_path = OUTPUTS_PATH / "45878514"
countries = ls(job_path)
all_countries = ls(job_path)
countries_by_cont = get_countries_by_continent(all_countries)
disp_posts = {c: get_param_vals_by_analysis("dispersion_proc", job_path / c) for c in countries}

In [ ]:
param = "dispersion_proc"
param_name = yml.safe_load(open(DATA_PATH / "config/priors.yml", "r"))["other"]["dispersion_proc"]["short_name"]
for cont, countries in countries_by_cont.items():
    cont_name = pc.convert_continent_code_to_continent_name(cont)
    display(Markdown(f"# {cont_name}"))
    title = f"Posterior distribution for {param_name} parameter, {cont_name}"
    param_posts = {}
    for country in countries:
        param_posts[country] = get_param_vals_by_analysis(param, job_path / country)
    if cont == "EU":
        eur_countries = list(param_posts.keys())
        display(plot_kde_comparison({k: param_posts[k] for k in eur_countries[:16]}, title))
        display(plot_kde_comparison({k: param_posts[k] for k in eur_countries[16:]}, title))
    else:
        display(plot_kde_comparison(param_posts, title))

# Best analysis
Best selected approach to analysis as determined
by the lowest average dispersion value needed in model fitting.
Google, green; Facebook tiles visited, blue; Facebook single tile, red.

In [ ]:
plt.style.use("default")
fig, ax, world = plot_world_country_outline()
ax.set_title("Best mobility type")

best_mob = {c: disp_posts[c].mean().idxmin() for c in countries}
world["best_mob"] = world["ISO_A3"].map(best_mob)
world["best_mob_colour"] = world["best_mob"].map(MOB_COLOURS | {"no_mob": "0.45"})
mob_avail = world[world["best_mob_colour"].notna()]
mob_unavail = world[world["best_mob_colour"].isna()]
mob_avail.plot(ax=ax, color=mob_avail["best_mob_colour"])
mob_unavail.plot(ax=ax, color="w", hatch="///", alpha=0.04)

# Improvement over no mobility analysis
The following plots show the proportion of analyses for which the
dispersion value was lower than a randomly selected equivalent run
from the analysis with no mobility data implemented.
Red colours indicate implementation of mobility reduced the 
dispersion parameter required for fitting.
Intermediate and blue colours indicate this did not occur.

In [ ]:
analysis_type = "g_mob"
display(Markdown(f"## {ANALYSIS_NAMES[analysis_type]}"))
prop_improve = get_prop_improve(disp_posts, analysis_type)
plot_prop_improve(prop_improve)

In [ ]:
analysis_type = "fb_visited_mob"
display(Markdown(f"## {ANALYSIS_NAMES[analysis_type]}"))
prop_improve = get_prop_improve(disp_posts, analysis_type)
plot_prop_improve(prop_improve)

In [ ]:
analysis_type = "fb_singletile_mob"
display(Markdown(f"## {ANALYSIS_NAMES[analysis_type]}"))
prop_improve = get_prop_improve(disp_posts, analysis_type)
plot_prop_improve(prop_improve)